# CSVLoader: CSV to RAG pipeline

This notebook turns rows from a CSV file into LangChain `Document` objects, chunks them, embeds them in FAISS, retrieves relevant rows, and asks a Groq LLM for a grounded answer.

## 1. Imports and project paths

`CSVLoader` is useful when knowledge is stored as spreadsheet-like rows. Each row retains metadata such as its source and row number, which helps trace a RAG answer back to its data.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_community.document_loaders import CSVLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter

def find_project_root() -> Path:
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / 'data').is_dir():
            return candidate
    raise FileNotFoundError('Run this notebook from the project root or its notebook folder.')

PROJECT_ROOT = find_project_root()
CSV_PATH = PROJECT_ROOT / 'data' / 'csv' / 'learning_topics.csv'
assert CSV_PATH.is_file(), f'CSV file not found: {CSV_PATH}'
print(f'Using CSV file: {CSV_PATH}')

## 2. Load and inspect CSV rows

In [ ]:
loader = CSVLoader(file_path=str(CSV_PATH), encoding='utf-8')
documents = loader.load()

print(f'Loaded {len(documents)} CSV rows as documents.')
print(documents[0].page_content)
print(documents[0].metadata)

## 3. Build the retrieval index

Chunking keeps each retrieved passage focused. `HuggingFaceEmbeddings` makes semantic vectors locally, and FAISS performs fast similarity search without requiring a separate database server.

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
vector_store = FAISS.from_documents(chunks, embeddings)
print(f'Indexed {len(chunks)} chunks in FAISS.')

## 4. Retrieve context and generate an answer

The LLM receives only retrieved context plus the question. This is the key RAG step: it makes the answer depend on your CSV instead of the model's general memory. Set `GROQ_API_KEY` in `.env` before running this cell.

In [ ]:
load_dotenv(PROJECT_ROOT / '.env')
groq_api_key = os.getenv('GROQ_API_KEY')
if not groq_api_key:
    raise RuntimeError('Set GROQ_API_KEY in .env before running the LLM cell.')

question = 'How do embeddings help a RAG application?'
retrieved_docs = vector_store.similarity_search(question, k=3)
context = '\n\n'.join(doc.page_content for doc in retrieved_docs)

prompt = PromptTemplate.from_template(
    'Answer only from the supplied context. If the context is insufficient, say so.\n\n'
    'Context:\n{context}\n\nQuestion: {question}\nAnswer:'
)
llm = ChatGroq(api_key=groq_api_key, model='openai/gpt-oss-20b', temperature=0.1, max_tokens=512)
answer = llm.invoke(prompt.format(context=context, question=question))

print(answer.content)
print('\nRetrieved sources:', [doc.metadata.get('source') for doc in retrieved_docs])